# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [1]:
# Optional setup: install dependencies if they are missing in your environment.
%pip install -q transformers torch


In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# sample_sentence = "TODO: replace with a short sentence you want to tokenize"
sample_sentence = "This is a short sentence I would like to tokenize."

print(sample_sentence)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

This is a short sentence I would like to tokenize.


In [3]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room #DONE.
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | this         |  2023
    2 | is           |  2003
    3 | a            |  1037
    4 | short        |  2460
    5 | sentence     |  6251
    6 | i            |  1045
    7 | would        |  2052
    8 | like         |  2066
    9 | to           |  2000
   10 | token        | 19204
   11 | ##ize        |  4697
   12 | .            |  1012
   13 | [SEP]        |   102
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (13, '[SEP]'), (14, '[PAD]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD]'), (19, '[PAD]'), (20, '[PAD]

### Exercise 1 reflection
- TODO: Describe how [CLS] and [SEP] behave inside the encoder.

DONE:

CLS behaves as a marker of document start, and SEP behaves as a marker of end of sentence.
- TODO: Explain how the attention mask hides padded positions from self-attention.

DONE:

The attention mask hides padded positions from self-attention by marking all padded positions as 0.
The padding choice made fits the sentence length as it fills up the remaining spaces after the sentence, period, and SEP to the set max length.


## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [4]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# sentence = "TODO: add a sentence whose sentiment you want to test"
sentence = "The movie was horrible. I suffered extremely through it."

prediction = sentiment_pipeline(sentence)
prediction


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'NEGATIVE', 'score': 0.9997888207435608}]

### Exercise 2 reflection
- TODO: Does the predicted label match your expectation? Why or why not?

DONE:

The predicted label matches my expectation because I inputted a clearly negative sentence to test it and it indeed returned a negative label with a very high score indicating that it's very negative.

- TODO: How confident is the model and what does the score tell you?

DONE:

The model is very confident. The score tells me that it is very confident; that the sentence was marked as very negative.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        '''TODO: load the tokenizer/model and move the model to the proper device.'''
        # 1. Load the tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.max_length = max_length

        # 2. Move model to the proper device (GPU if available, else CPU)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

        # Set model to evaluation mode
        self.model.eval()

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        '''TODO: clean the text, tokenize, and return tensors ready for inference.'''
        # Clean text (optional, but good practice to strip whitespace)
        text = text.strip()

        # Tokenize and return tensors (return_tensors="pt" for PyTorch)
        inputs = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Move the tensors to the same device as the model
        return {k: v.to(self.device) for k, v in inputs.items()}

    def predict(self, text: str) -> Dict[str, float]:
        '''TODO: run a forward pass, apply softmax, and return a label plus probability.'''
        # 1. Preprocess the input
        inputs = self.preprocess(text)

        # 2. Run a forward pass (no gradients needed for inference)
        with torch.no_grad():
            outputs = self.model(**inputs)

        # 3. Apply Softmax to get probabilities
        # outputs.logits is [batch_size, num_labels]
        probabilities = F.softmax(outputs.logits, dim=-1)

        # 4. Extract the top label and probability
        # In SST-2: index 0 is NEGATIVE, index 1 is POSITIVE
        score, index = torch.max(probabilities, dim=-1)
        label = self.model.config.id2label[index.item()]

        return {
            "label": label,
            "probability": score.item()
        }

In [15]:
# TODO: instantiate your analyzer and test several sentences once the class is ready.
analyzer = BERTSentimentAnalyzer()
samples = [
#     "TODO: add a clearly positive statement",
#     "TODO: add a clearly negative statement"
"What an amazing movie! I loved it.",
"I hated this movie. What a waste of time and money."
]
for text in samples:
    print(text)
    print(analyzer.predict(text))


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

What an amazing movie! I loved it.
{'label': 'POSITIVE', 'probability': 0.9998857975006104}
I hated this movie. What a waste of time and money.
{'label': 'NEGATIVE', 'probability': 0.999804675579071}


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [1]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        # 1. Load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)

        # 2. Detect device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

        # 3. Use the 'pipeline' utility for easier word-piece merging
        # The 'aggregation_strategy' handles merging "word pieces" back into whole words
        self.ner_pipeline = pipeline(
            "ner",
            model=self.model,
            tokenizer=self.tokenizer,
            device=0 if torch.cuda.is_available() else -1,
            aggregation_strategy="simple"
        )

    def recognize(self, text: str):
        # The pipeline handles tokenization, forward pass, BIO mapping,
        # and merging word pieces automatically.
        entities = self.ner_pipeline(text)

        # Return structured entities
        return entities

In [2]:
# TODO: instantiate the recognizer and test it on text that includes people, places, or organizations.
ner = BERTNamedEntityRecognizer()
# sample_text = "TODO: add a short paragraph with at least two entities."
sample_text = "This is a short paragraph with at least two entities, one positive and one negative. I've seen several movies recently. I loved the Lion King. However, I hated Terminator 3."

ner.recognize(sample_text)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'MISC',
  'score': np.float32(0.9741228),
  'word': 'Lion King',
  'start': 132,
  'end': 141},
 {'entity_group': 'MISC',
  'score': np.float32(0.99274683),
  'word': 'Terminator 3',
  'start': 160,
  'end': 172}]

TODO: Explain how you handled subword tokens that begin with ##.

A token that starts with ## belongs to the previous token. We must simply concatenate the characters and average the confidence scores of the fragments.

As shown in the previous code, the pipeline with aggregation_strategy="simple" does this for us automatically.

## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Bidirectional; processes both the left and right sides of a sequence to estimate a masked token. | Autoregressive; processes a sequence from left to right, estimating the next token. |
| Primary purpose | Understanding context through masked language modeling. | Generating text by predicting the next token in a sequence. |
| Typical use cases | Sentiment analysis, NER, and question answering. | Creative writing, chatbots, and code generation. |
| Strengths | Superior at capturing nuances of context from both sides. | Excellent at maintaining flow and generating coherent long-form text. |
| Weaknesses | Cannot generate text effectively or handle open-ended tasks. | Only "sees" previous tokens, missing future context in a sentence. |


## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. TODO: Describe how BERT encodes queries and documents.

BERT functions as a Bi-Encoder. Every document is passed through BEFT to create a fixed-length numerical vector (an "embedding") that captures its semantic meaning. When a query is provided, it is also converted into a vector using the same model. Because BErT is bidirectional, the embeddings represent the text's deep context rather than merely matching keywords.

2. TODO: Explain how those embeddings are stored and searched in a vector database.

When documents are transformed into vectors, they are stored in a specialized vector database (such as Pinecone, Milvus, or FAISS). The database performs a similarity search, calculating the distance between the query vector and all document vectors. This enables it to find relevant passages even if the user uses different vocabulary than the source text.

3. TODO: Outline how the retrieved passages are handed to a generative model like GPT.

After the vector database identifies the $k$ most relevant passages (the "retrieved" data), they are combined with the user's original query into a single enlarged prompt. This prompt essentially tells the generative model (such as GPT): "Using only this information, please answer the user's question." The generative model then uses its linguistic fluency to synthesize a natural-sounding answer based strictly on the factual evidence provided by the BERT retrieval step.

4. TODO: Provide a concrete application example (industry or product) where RAG with BERT makes sense.

One good use case for BERT-powered RAG is a customer support bot for a technical hardware company. The company may have a lot of text in manuals and troubleshooting guides. BERT can encode these documents to understand the relationship between a user-provided problem and a solution, even if the documents use different phrasing. When a user asks a question, the RAG system retrieves the exact technical fix from the database and uses GPT to explain it to the user in a friendly, conversational tone.